# Product Gross Price ETL Pipeline (Medallion Architecture)

This notebook implements an automated PySpark ETL pipeline that ingests, cleanses, standardizes, and merges gross price data across Bronze, Silver, and Gold Delta layers within Databricks.

WARNING
The Delta Merge (Upsert) operation updates existing records and inserts new ones based on `product_code`. Ensure that missing or zero prices are properly filtered or ranked before running the merge to prevent overwriting valid price attributes with invalid values.

## 1. Environment & Dynamic Parameter Initialization
Configures notebook parameters (widgets) for catalog and data source names to enable dynamic runtime execution.

In [0]:
# Import PySpark window functions, standard functions, and Delta table modules
from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "../Utitlites"

In [0]:
# ==========================================
# 2. Dynamic Notebook Parameters (Widgets)
# ==========================================
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_Sourse', 'customers', 'Data Source')

# Extract parameter values
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_Sourse')

In [0]:
# Read raw CSV file and attach metadata attributes
df = (spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(f'{path_file}{data_source}.csv')
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', '_metadata.file_name', '_metadata.file_size')
)

# Display Schema
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### 2. Processing to Bronze Layer

In [0]:
# Save raw data to Bronze Delta Table
df.write\
    .format('delta')\
    .mode('overwrite')\
    .option('delta.enableChangeDataFeed', 'true')\
    .saveAsTable(f'{catalog}.bronze.sb_dim_{data_source}')

# Show the Bronze Table
df_bronze = spark.sql(f'SELECT * FROM {catalog}.bronze.sb_dim_{data_source}')
df_bronze.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025/07/01|        -84|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891101|01/08/2025|    unknown|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891101|2025/09/01|         84|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891101|2025-10-01|         83|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891101|2025-11-01|         83|2026-10-08 11:40:...|gross_price.csv|     2741|
|  88888888|2025-12-01|        -83|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891102|2025-07-01|         68|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891102|2025-08-01|         68|2026-10-08 11:40:...|gross_price.csv|     2741|
|  25891102|2025-09-01|         68|2026-10-08 11:40:...|gross_price.csv|     2741|
|  2

### Cleaning Stage

In [0]:
# Parse string dates into unified Date format across multiple input patterns
df_bronze = df_bronze.withColumn(
    'month', 
    F.coalesce(
        F.try_to_date(F.col("month"), 'yyyy/MM/dd'),
        F.try_to_date(F.col("month"), 'dd/MM/yyyy'),
        F.try_to_date(F.col("month"), 'yyyy-MM-dd'),
        F.try_to_date(F.col("month"), 'dd-MM-yyyy')
    )
)

df_bronze.select("month").show()

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
+----------+
only showing top 20 rows


In [0]:
# Clean numeric price field, convert valid values to absolute double, and fallback invalid values to 0
df_bronze = df_bronze.withColumn(
    'gross_price',
    F.when(
        F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
        F.abs(F.col("gross_price").cast('double'))
    ).otherwise(0)
)

df_bronze.select("gross_price").show()

+-----------+
|gross_price|
+-----------+
|       84.0|
|        0.0|
|       84.0|
|       83.0|
|       83.0|
|       83.0|
|       68.0|
|       68.0|
|       68.0|
|       69.0|
|       69.0|
|       69.0|
|       85.0|
|       85.0|
|       85.0|
|        0.0|
|       86.0|
|       86.0|
|      108.0|
|      108.0|
+-----------+
only showing top 20 rows


In [0]:
# Load product dimension table to map product_code by product_id
sb_products = spark.sql(f'SELECT product_code, product_id FROM {catalog}.gold.sb_dim_products')

# Join Bronze dataframe with product dimension table
df_bronze = df_bronze.join(sb_products, on='product_id', how='inner')

# Show the Bronze Table
df_bronze.show()

+----------+----------+-----------+--------------------+---------------+---------+--------------------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|        product_code|
+----------+----------+-----------+--------------------+---------------+---------+--------------------+
|  25891101|2025-07-01|       84.0|2026-10-08 11:40:...|gross_price.csv|     2741|e91ba9d665f90254d...|
|  25891101|2025-08-01|        0.0|2026-10-08 11:40:...|gross_price.csv|     2741|e91ba9d665f90254d...|
|  25891101|2025-09-01|       84.0|2026-10-08 11:40:...|gross_price.csv|     2741|e91ba9d665f90254d...|
|  25891101|2025-10-01|       83.0|2026-10-08 11:40:...|gross_price.csv|     2741|e91ba9d665f90254d...|
|  25891101|2025-11-01|       83.0|2026-10-08 11:40:...|gross_price.csv|     2741|e91ba9d665f90254d...|
|  25891102|2025-07-01|       68.0|2026-10-08 11:40:...|gross_price.csv|     2741|e92c739a8d78cd6cb...|
|  25891102|2025-08-01|       68.0|2026-10-08 11:40:...|gross_pr

In [0]:
# Derive transaction year from month date
df_bronze = df_bronze.withColumn('year', F.year(F.col('month')))

# Standardize price column name to price_inr
if 'gross_price' in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed('gross_price', 'price_inr')

# Flag zero or missing price records
df_bronze = df_bronze.withColumn('is_zero', F.when(F.col('price_inr') == 0, 1).otherwise(0))

# Show Results
df_bronze.select('year', 'price_inr', 'is_zero').show()

+----+---------+-------+
|year|price_inr|is_zero|
+----+---------+-------+
|2025|     84.0|      0|
|2025|      0.0|      1|
|2025|     84.0|      0|
|2025|     83.0|      0|
|2025|     83.0|      0|
|2025|     68.0|      0|
|2025|     68.0|      0|
|2025|     68.0|      0|
|2025|     69.0|      0|
|2025|     69.0|      0|
|2025|     69.0|      0|
|2025|     85.0|      0|
|2025|     85.0|      0|
|2025|     85.0|      0|
|2025|      0.0|      1|
|2025|     86.0|      0|
|2025|     86.0|      0|
|2025|    108.0|      0|
|2025|    108.0|      0|
|2025|    108.0|      0|
+----+---------+-------+
only showing top 20 rows


In [0]:
# Define window specification partitioned by product_code and year, prioritized by non-zero price and latest month
w = (
    Window.partitionBy('product_code', 'year')
          .orderBy(F.col('is_zero'), F.col('month').desc())
)

# Keep top-ranked valid record per product per year
df_bronze = (
    df_bronze
    .withColumn('rank', F.row_number().over(w))
    .filter(F.col('rank') == 1)
)

# Show Results
df_bronze.show()

+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|product_id|     month|price_inr|      read_timestamp|      file_name|file_size|        product_code|year|is_zero|rank|
+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|  25891503|2025-12-01|    281.0|2026-10-08 11:40:...|gross_price.csv|     2741|062f5574bbdf4386b...|2025|      0|   1|
|  25891103|2025-12-01|     86.0|2026-10-08 11:40:...|gross_price.csv|     2741|102628255d24304d6...|2025|      0|   1|
|  25891301|2025-12-01|    493.0|2026-10-08 11:40:...|gross_price.csv|     2741|3cab59f0592428527...|2025|      0|   1|
|  25891603|2025-12-01|    187.0|2026-10-08 11:40:...|gross_price.csv|     2741|451f7167b28a25bde...|2025|      0|   1|
|  25891601|2025-11-01|    440.0|2026-10-08 11:40:...|gross_price.csv|     2741|716fa4e54b7894c91...|2025|      0|   1|
|  25891602|2025-12-01|    296.0|2026-10

## Processing To Silver Layer

In [0]:
# Save cleansed dataframe to Silver Delta Table
df_bronze.write \
    .format('delta') \
    .mode('overwrite') \
    .option('delta.enableChangeDataFeed', 'true') \
    .saveAsTable(f'{catalog}.silver.sb_dim_{data_source}')

# Load Silver Delta Table
df_silver = spark.sql(f'SELECT * FROM {catalog}.silver.sb_dim_{data_source}')
df_silver.show()

+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|product_id|     month|price_inr|      read_timestamp|      file_name|file_size|        product_code|year|is_zero|rank|
+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|  25891503|2025-12-01|    281.0|2026-10-08 11:40:...|gross_price.csv|     2741|062f5574bbdf4386b...|2025|      0|   1|
|  25891103|2025-12-01|     86.0|2026-10-08 11:40:...|gross_price.csv|     2741|102628255d24304d6...|2025|      0|   1|
|  25891301|2025-12-01|    493.0|2026-10-08 11:40:...|gross_price.csv|     2741|3cab59f0592428527...|2025|      0|   1|
|  25891603|2025-12-01|    187.0|2026-10-08 11:40:...|gross_price.csv|     2741|451f7167b28a25bde...|2025|      0|   1|
|  25891601|2025-11-01|    440.0|2026-10-08 11:40:...|gross_price.csv|     2741|716fa4e54b7894c91...|2025|      0|   1|
|  25891602|2025-12-01|    296.0|2026-10

## Processing To Gold

In [0]:
# Save cleansed dataframe to Silver Delta Table
df_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('delta.enableChangeDataFeed', 'true')\
    .saveAsTable(f'{catalog}.gold.sb_dim_{data_source}')

# Load Silver Delta Table
df_silver = spark.sql(f'SELECT * FROM {catalog}.gold.sb_dim_{data_source}')
df_silver.show()

+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|product_id|     month|price_inr|      read_timestamp|      file_name|file_size|        product_code|year|is_zero|rank|
+----------+----------+---------+--------------------+---------------+---------+--------------------+----+-------+----+
|  25891503|2025-12-01|    281.0|2026-10-08 11:40:...|gross_price.csv|     2741|062f5574bbdf4386b...|2025|      0|   1|
|  25891103|2025-12-01|     86.0|2026-10-08 11:40:...|gross_price.csv|     2741|102628255d24304d6...|2025|      0|   1|
|  25891301|2025-12-01|    493.0|2026-10-08 11:40:...|gross_price.csv|     2741|3cab59f0592428527...|2025|      0|   1|
|  25891603|2025-12-01|    187.0|2026-10-08 11:40:...|gross_price.csv|     2741|451f7167b28a25bde...|2025|      0|   1|
|  25891601|2025-11-01|    440.0|2026-10-08 11:40:...|gross_price.csv|     2741|716fa4e54b7894c91...|2025|      0|   1|
|  25891602|2025-12-01|    296.0|2026-10

In [0]:
# Reference target Gold dimension table
parent_table_name = f"{catalog}.gold.dim_{data_source}"

# 1. Deduplicate existing Gold table records if table exists
if spark.catalog.tableExists(parent_table_name):
    spark.read.table(parent_table_name) \
        .dropDuplicates(["product_code"]) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(parent_table_name)

# 2. Get DeltaTable instance correctly for the MERGE operation
parent_table = DeltaTable.forName(spark, parent_table_name)

# 3. Query source staging table
child_table = spark.sql(f'''
    SELECT product_code, 
           price_inr, 
           year 
    FROM {catalog}.gold.sb_dim_{data_source}
''')

# 4. Merge staged updates into target Gold table
parent_table.alias('target').merge(
    source=child_table.alias('source'),
    condition='target.product_code = source.product_code'
).whenMatchedUpdate(
    set={
        'price_inr': 'source.price_inr',
        'year': 'source.year'
    }
).whenNotMatchedInsert(
    values={
        'product_code': 'source.product_code',
        'price_inr': 'source.price_inr',
        'year': 'source.year'
    }
).execute()

# Preview updated Gold dimension table
spark.sql(f'''
    SELECT *
    FROM {catalog}.gold.dim_products
    WHERE LENGTH(product_code) = 64
''').show()

1. LAYER 1: BRONZE LAYER VALIDATION (Source Audit)
[-] Raw duplicate records found in Bronze: 3
[-] Records with invalid or non-numeric price format: 6
[-] Records with missing/null dates in Bronze: 0

2. LAYER 2: SILVER LAYER VALIDATION (Cleansing & Normalization)
[-] Records with zero or null price in Silver: 0
[-] Records missing product_code key mapping: 0
[-] Non-deduplicated records remaining in Silver (Should be 0): 0

3. LAYER 3: GOLD LAYER VALIDATION (Dimensional Integrity)
[-] Duplicate Primary Key (product_code + year) in Gold Table (Should be 0): 0
[-] Total processed records in Silver: 17
[-] Total records in Gold Dimension Table: 414

4. FINAL GOLD TABLE PREVIEW
+----------------------------------------------------------------+---------+----+
|product_code                                                    |price_inr|year|
+----------------------------------------------------------------+---------+----+
|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|281

### Check Data Quality

In [0]:
print("=" * 60)
print("1. LAYER 1: BRONZE LAYER VALIDATION (Source Audit)")
print("=" * 60)

df_bronze_audit = spark.sql(f'SELECT * FROM {catalog}.bronze.sb_dim_{data_source}')

bronze_duplicates = df_bronze_audit.groupBy("product_id", "month").count().filter("count > 1").count()
print(f"[-] Raw duplicate records found in Bronze: {bronze_duplicates}")

invalid_prices = df_bronze_audit.filter(~F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$')).count()
print(f"[-] Records with invalid or non-numeric price format: {invalid_prices}")

unparsed_dates = df_bronze_audit.filter(F.col("month").isNull()).count()
print(f"[-] Records with missing/null dates in Bronze: {unparsed_dates}\n")


print("=" * 60)
print("2. LAYER 2: SILVER LAYER VALIDATION (Cleansing & Normalization)")
print("=" * 60)

df_silver_audit = spark.table(f"{catalog}.silver.sb_dim_{data_source}")

zero_prices = df_silver_audit.filter((F.col("price_inr").isNull()) | (F.col("price_inr") == 0)).count()
print(f"[-] Records with zero or null price in Silver: {zero_prices}")

unmapped_products = df_silver_audit.filter(F.col("product_code").isNull()).count()
print(f"[-] Records missing product_code key mapping: {unmapped_products}")

duplicate_ranks = df_silver_audit.filter(F.col("rank") > 1).count()
print(f"[-] Non-deduplicated records remaining in Silver (Should be 0): {duplicate_ranks}\n")


print("=" * 60)
print("3. LAYER 3: GOLD LAYER VALIDATION (Dimensional Integrity)")
print("=" * 60)

df_gold_audit = spark.table(f"{catalog}.gold.dim_{data_source}")

pk_duplicates = df_gold_audit.groupBy("product_code", "year").count().filter("count > 1").count()
print(f"[-] Duplicate Primary Key (product_code + year) in Gold Table (Should be 0): {pk_duplicates}")

silver_count = df_silver_audit.count()
gold_count = df_gold_audit.count()
print(f"[-] Total processed records in Silver: {silver_count}")
print(f"[-] Total records in Gold Dimension Table: {gold_count}\n")

print("=" * 60)
print("4. FINAL GOLD TABLE PREVIEW")
print("=" * 60)

spark.sql(f'''
    SELECT product_code, price_inr, year
    FROM {catalog}.gold.dim_{data_source}
    WHERE LENGTH(product_code) = 64
    LIMIT 10
''').show(truncate=False)

1. LAYER 1: BRONZE LAYER VALIDATION (Source Audit)
[*] Raw duplicate records found in Bronze: 3
[*] Records with invalid or non-numeric price format: 6
[*] Records with missing/null dates in Bronze: 0

2. LAYER 2: SILVER LAYER VALIDATION (Cleansing & Normalization)
[*] Records with zero or null price in Silver: 0
[*] Records missing product_code key mapping: 0
[*] Non-deduplicated records remaining in Silver (Should be 0): 0

3. LAYER 3: GOLD LAYER VALIDATION (Dimensional Integrity)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5166602358600116>, line 53
     50 df_gold_audit = spark.table(f"{catalog}.gold.dim_{data_source}")
     52 # Check 3.1: Primary Key Uniqueness (Should be 0)
---> 53 pk_duplicates = df_gold.groupBy("product_code", "year").count().filter("count > 1").count()
     54 print(f"[*] Duplicate Primary Key (product_code) in Gold Table (Should be 0): {pk_duplicates}")
     56 # Check 3.2: Record Reconciliation

NameError: name 'df_gold' is not defined